# Tutorial 02 — Analysis & DNA

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/vectrix/blob/master/notebooks/tutorials/02_analyze.ipynb)

**Understand your data before you forecast it.** Every time series has a unique statistical fingerprint — its "DNA." Vectrix extracts 65+ features to build a profile that reveals trend, seasonality, volatility, structural breaks, and forecasting difficulty.

| What you'll learn | Time |
|:------------------|:-----|
| Basic analysis | 1 min |
| DNA profiling | 2 min |
| Changepoints & anomalies | 1 min |
| Data characteristics | 2 min |
| Analyze-then-forecast pattern | 1 min |

In [ ]:
!pip install -q vectrix

## 1. Basic Analysis

The `analyze()` function accepts the same input formats as `forecast()` — lists, arrays, DataFrames, Series, or CSV paths.

In [ ]:
from vectrix import analyze, loadSample

df = loadSample("airline")
report = analyze(df, date="date", value="passengers")
print(report.summary())

In [ ]:
# Also works with raw lists
data = [120, 135, 148, 130, 155, 170, 162, 180, 195, 185, 200, 215]
report_list = analyze(data)
print(report_list.summary())

## 2. DNA Profile

The DNA profile determines which models to prioritize, how to set hyperparameters, and what difficulty to expect.

In [ ]:
report = analyze(df, date="date", value="passengers")
dna = report.dna

print(f"Fingerprint:  {dna.fingerprint}")
print(f"Difficulty:   {dna.difficulty} ({dna.difficultyScore:.0f}/100)")
print(f"Category:     {dna.category}")
print(f"Recommended:  {dna.recommendedModels[:5]}")

### DNA Attributes

| Attribute | Type | Description |
|-----------|------|-------------|
| `fingerprint` | `str` | Deterministic hash — identical data always produces the same value |
| `difficulty` | `str` | `easy`, `medium`, `hard`, or `very_hard` |
| `difficultyScore` | `float` | Numeric score (0-100, higher = harder) |
| `category` | `str` | `seasonal`, `trending`, `volatile`, `intermittent`, `stationary` |
| `recommendedModels` | `list[str]` | Ordered list of optimal models |

## 3. Changepoints & Anomalies

**Changepoints** are locations where the statistical properties shift (mean, variance, trend slope).

**Anomalies** are isolated spikes or dips — data errors, one-time events, or external shocks.

In [ ]:
print(f"Changepoints at indices: {report.changepoints}")
print(f"Anomaly indices:         {report.anomalies}")
print(f"Number of anomalies:     {len(report.anomalies)}")

## 4. Data Characteristics

A comprehensive statistical profile: trend direction, seasonal patterns, volatility, and predictability.

In [ ]:
c = report.characteristics

print(f"Length:         {c.length}")
print(f"Period:         {c.period}")
print(f"Trend:          {c.hasTrend} ({c.trendDirection}, strength {c.trendStrength:.2f})")
print(f"Seasonality:    {c.hasSeasonality} (strength {c.seasonalStrength:.2f})")
print(f"Volatility:     {c.volatilityLevel} ({c.volatility:.4f})")
print(f"Predictability: {c.predictabilityScore}/100")
print(f"Outliers:       {c.outlierCount} ({c.outlierRatio:.1%})")

## 5. Extracted Features (65+)

Access the raw statistical features that power the DNA classification.

In [ ]:
features = report.features

print(f"Total features: {len(features)}")
print()
for key, value in list(features.items())[:10]:
    print(f"  {key}: {value}")

Key feature names: `trendStrength`, `seasonalStrength`, `seasonalPeakPeriod`, `hurstExponent`, `volatilityClustering`, `nonlinearAutocorr`, `demandDensity`, `spectralEntropy`, `approximateEntropy`, `forecastability`

## 6. Analyze Before Forecasting

A powerful pattern: analyze first, then adjust your strategy based on what the DNA reveals.

In [ ]:
from vectrix import analyze, forecast, loadSample

df = loadSample("retail")
report = analyze(df, date="date", value="sales")

print(f"Category: {report.dna.category}")
print(f"Difficulty: {report.dna.difficulty}")
print(f"Changepoints: {len(report.changepoints)}")
print(f"Anomalies: {len(report.anomalies)}")
print()

if report.dna.difficulty == "very_hard":
    print("Warning: This series is very hard to forecast.")
    print(f"Consider using: {report.dna.recommendedModels[:3]}")

if report.characteristics.hasSeasonality:
    period = report.characteristics.period
    print(f"Seasonal period: {period}")
    result = forecast(df, date="date", value="sales", steps=period)
else:
    result = forecast(df, date="date", value="sales", steps=14)

print(f"\nModel: {result.model}, MAPE: {result.mape:.2f}%")

## 7. Direct ForecastDNA Access

For lower-level control — custom model selection logic or caching DNA profiles.

In [ ]:
from vectrix import ForecastDNA
import numpy as np

dna = ForecastDNA()
data = np.random.randn(200).cumsum() + 100
profile = dna.analyze(data, period=7)

print(f"Fingerprint: {profile.fingerprint}")
print(f"Difficulty:  {profile.difficulty} ({profile.difficultyScore:.0f}/100)")
print(f"Recommended: {profile.recommendedModels[:5]}")

## 8. Complete Example

In [ ]:
from vectrix import analyze, loadSample

df = loadSample("energy")
report = analyze(df, date="date", value="consumption_kwh")

print("=== DNA ===")
print(f"Category:   {report.dna.category}")
print(f"Difficulty: {report.dna.difficulty}")
print(f"Top models: {report.dna.recommendedModels[:3]}")

print()
print("=== Characteristics ===")
c = report.characteristics
print(f"Trend:    {c.trendDirection} (strength {c.trendStrength:.2f})")
print(f"Seasonal: period={c.period}, strength={c.seasonalStrength:.2f}")
print(f"Predictability: {c.predictabilityScore}/100")

print()
print("=== Structural Breaks ===")
print(f"Changepoints: {report.changepoints}")
print(f"Anomalies:    {report.anomalies}")

---

**Next:** [Tutorial 03 — Regression](03_regression.ipynb)